***IMPORT LIBRARIES***

***Import Pandas,Numpy***

In [2]:
import pandas as pd
import numpy as np

***IMPORT SCIKIT-LEARN MODULES***

***Import Scikit-Learn modules for Data Preprocessing,Building the Model,Evaluating Model***



In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    accuracy_score
)

***DATA LOADING AND INITIAL VALIDATION***

***1.Import MySQL Connection***

In [4]:
import sklearn 
from sqlalchemy import create_engine

In [5]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv(r"C:\Users\LP\Desktop\python\.env")
user=os.getenv("DB_USER")
password=os.getenv("DB_PASSWORD")
host=os.getenv("DB_HOST")
port=os.getenv("DB_PORT")
database=os.getenv("DB_NAME")

encoded_password=quote_plus(password)

engine=create_engine(
    f"mysql+pymysql://{user}:{encoded_password}@{host}:{port}/{database}"
)


***2.Load Data into Python***

In [7]:

query= "select * from pd_risk_default"
df=pd.read_sql(query,engine)

***3. Validate the Dataset***

In [8]:
df.head()

,age,city_tier,state,emp_type,vintage_years,monthly_income,existing_loans_count,existing_emi_monthly,credit_inquiries,loan_amount,loan_tenure,loan_purpose,collateral_provided,cibil_score,default_flag,emi_to_income
0,22,Tier 3,Haryana,Salaried - PSU,0.5,12000,1,1700.0,2.0,149000,48,Personal Expense,No,649,0,14.17
1,35,Tier 2,Gujarat,Salaried - Private,0.5,32500,3,11500.0,0.0,156000,60,Personal Expense,No,538,0,35.38
2,41,Tier 2,Uttar Pradesh,Salaried - Government,2.8,52500,0,6000.0,0.0,70000,36,Wedding,No,636,0,11.43
3,29,Tier 2,Kerala,Salaried - PSU,0.7,15500,1,1900.0,0.0,108000,48,Education,No,721,0,12.26
4,31,Tier 1,Punjab,Self-Employed Professional,4.7,269600,2,50800.0,0.0,466000,24,Education,No,586,0,18.84


In [48]:
df.shape


(25000, 16)

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   25000 non-null  int64  
 1   city_tier             25000 non-null  str    
 2   state                 25000 non-null  str    
 3   emp_type              25000 non-null  str    
 4   vintage_years         24625 non-null  float64
 5   monthly_income        25000 non-null  int64  
 6   existing_loans_count  25000 non-null  int64  
 7   existing_emi_monthly  24500 non-null  float64
 8   credit_inquiries      24750 non-null  float64
 9   loan_amount           25000 non-null  int64  
 10  loan_tenure           25000 non-null  int64  
 11  loan_purpose          25000 non-null  str    
 12  collateral_provided   25000 non-null  str    
 13  cibil_score           25000 non-null  int64  
 14  default_flag          25000 non-null  int64  
 15  emi_to_income         24500 no

***4. Check Missing Values***

In [9]:
df.isnull().sum()

age                       0
city_tier                 0
state                     0
emp_type                  0
vintage_years           375
monthly_income            0
existing_loans_count      0
existing_emi_monthly    500
credit_inquiries        250
loan_amount               0
loan_tenure               0
loan_purpose              0
collateral_provided       0
cibil_score               0
default_flag              0
emi_to_income           500
dtype: int64

***5. Analyse Target distribution***

In [ ]:
default_proportion=df["default_flag"].value_counts(normalize=True)*100
default_proportion


default_flag
0    93.86
1     6.14
Name: proportion, dtype: float64

 ***DATA SPLITTING***
 
 ***6. Split data into Features and target***

In [11]:
X=df.drop("default_flag",axis=1)
Y=df["default_flag"]

***7. Split data into Training and Testing datasets***

In [14]:
X_train,X_test,Y_train,Y_test=train_test_split(
    X,Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

***DATA PREPROCESSING***

***8. Identify Numerical and Categorical Features***

In [16]:
num_cols=[
    "age",
    "vintage_years",
    "monthly_income",
    "existing_loans_count",
    "existing_emi_monthly",
    "credit_inquiries",
    "loan_amount",
    "loan_tenure",
    "cibil_score",
    "emi_to_income"
]
categorical_cols=[
    "loan_purpose",
    "collateral_provided",
    "city_tier",
    "state",
    "emp_type",
    
]

***9. Impute and Scale Numerical Features***

In [17]:
numeric_pipeline= Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("Scaler",StandardScaler())

])

***10. Impute and Encode categorical Features***

In [18]:
categorical_pipeline=Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

***11. Combine Preprocessing with Column Transformer***

In [19]:
preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,num_cols),
    ("cat",categorical_pipeline,categorical_cols)
])

***MODEL BUILDING***

***12. Build Logistic Regression pipeline***

In [20]:
model=Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",LogisticRegression(max_iter=1000))
])

***13. Train the Model***

In [36]:
model.fit(X_train,Y_train);

***MODEL EVALUATION***

***14. Generate Default Probabilities and Predictions***

In [37]:
Y_prob=model.predict_proba(X_test)[:,1]
Y_prob[:5]

array([0.04048774, 0.08749132, 0.32290817, 0.00232406, 0.01855406])

In [38]:
Y_pred=model.predict(X_test)
Y_pred[:5]

array([0, 0, 0, 0, 0])

***15. Evaluate Model Performance***

***A. Roc-auc and Pr-auc scores***

In [24]:
auc_score=roc_auc_score(Y_test,Y_prob)
print("ROC-AUC:",auc_score)

ROC-AUC: 0.8657117017444373


In [25]:
pr_auc=average_precision_score(Y_test,Y_prob)
print("PR-AUC:",pr_auc)

PR-AUC: 0.37052102633386463


***B. Confusion Matrix***

In [46]:
print(confusion_matrix(Y_test,Y_pred))



[[4665   28]
 [ 276   31]]


***C. Classification Report***

In [27]:
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.94      0.99      0.97      4693
           1       0.53      0.10      0.17       307

    accuracy                           0.94      5000
   macro avg       0.73      0.55      0.57      5000
weighted avg       0.92      0.94      0.92      5000



***THRESHOLD ANALYSIS***

***16.Evaluate Model performance Across Different Probability Cutoffs***

In [39]:
thresholds=[0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80]
results=[]
for threshold in thresholds:
    Y_pred_threshold=(Y_prob >=threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(Y_test,Y_pred_threshold),
        "Recall": recall_score(Y_test,Y_pred_threshold),
        "F1": f1_score(Y_test,Y_pred_threshold),
        "Accuracy":accuracy_score(Y_test,Y_pred_threshold)
    })
threshold_table=pd.DataFrame(results)
print(threshold_table)

   Threshold  Precision    Recall        F1  Accuracy
0        0.1   0.249708  0.697068  0.367698    0.8528
1        0.2   0.388451  0.482085  0.430233    0.9216
2        0.3   0.495000  0.322476  0.390533    0.9382
3        0.4   0.518519  0.182410  0.269880    0.9394
4        0.5   0.525424  0.100977  0.169399    0.9392
5        0.6   0.714286  0.065147  0.119403    0.9410
6        0.7   0.636364  0.022801  0.044025    0.9392
7        0.8   0.500000  0.003257  0.006472    0.9386


***17. Confusion Matrix at Selected Cutoff***

In [40]:
final_threshold=0.20
Y_pred_20=(Y_prob>=final_threshold).astype(int)
print(confusion_matrix(Y_test,Y_pred_20))


[[4460  233]
 [ 159  148]]


***RISK SEGMENTATION***

***19. Convert Predicted Default Probabilities into Risk Segments***


In [31]:
import pandas as pd
risk_band=pd.cut(
  Y_prob,
  bins=[0,0.10,0.20,0.40,1.00],
  labels=["Low Risk","Medium Risk","High Risk","Very High Risk"],
  include_lowest=True
 )
risk_band.value_counts().sort_index

<bound method Series.sort_index of Low Risk          4143
Medium Risk        476
High Risk          273
Very High Risk     108
Name: count, dtype: int64>

***PORTFOLIO SCORING***

***1. Generate PD for the full dataset***

In [44]:
X_full=df.drop("default_flag",axis=1)
df_scored=df.copy()
df_scored["PD"]=model.predict_proba(X_full)[:, 1]
df_scored.head()
df_scored[['monthly_income','cibil_score','loan_amount','emi_to_income','PD']].head(5)

,monthly_income,cibil_score,loan_amount,emi_to_income,PD
0,12000,649,149000,14.17,0.015275
1,32500,538,156000,35.38,0.185618
2,52500,636,70000,11.43,0.012091
3,15500,721,108000,12.26,0.001907
4,269600,586,466000,18.84,0.012914


***2. Create Risk Segments for full dataset***

In [45]:
df_scored["risk_band"]=pd.cut(
  df_scored["PD"],
  bins=[0,0.10,0.20,0.40,1.00],
  labels=["Low Risk","Medium Risk","High Risk","Very High Risk"],
  include_lowest=True
 )
df_scored[["default_flag","PD","risk_band"]].head(5)

,default_flag,PD,risk_band
0,0,0.015275,Low Risk
1,0,0.185618,Medium Risk
2,0,0.012091,Low Risk
3,0,0.001907,Low Risk
4,0,0.012914,Low Risk


In [34]:
df_scored["risk_band"].value_counts()

risk_band
Low Risk          20741
Medium Risk        2379
High Risk          1349
Very High Risk      531
Name: count, dtype: int64

***3. Export for Power BI***

In [35]:
df_scored.to_csv("pd_credit_risk_full.csv",index=False)